In [1]:
import duckdb
con = duckdb.connect("data/flights.duckdb")

In [34]:
model_df = con.execute('''SELECT flight_date, carrier_code, origin_airport, dest_airport, dep_hour, distance, day_of_week, is_weekend, is_holiday, delayed FROM fact_flights
 LEFT JOIN dim_date ON flight_date = date WHERE delayed IS NOT NULL''').df()
display(model_df)
assert(0.2 < model_df["delayed"].mean() < 0.25)
print("Average Delay Rate:", model_df["delayed"].mean())
model_df.isna().sum() # We have no nulls


,flight_date,carrier_code,origin_airport,dest_airport,dep_hour,distance,day_of_week,is_weekend,is_holiday,delayed
0,2025-10-16,UA,CLE,EWR,12,404.0,4,False,False,0
1,2025-10-16,UA,ORD,CLE,7,316.0,4,False,False,0
2,2025-10-16,UA,DEN,CHS,10,1449.0,4,False,False,1
3,2025-10-16,UA,SBA,DEN,5,916.0,4,False,False,0
4,2025-10-16,UA,BTV,ORD,16,763.0,4,False,False,0
...,...,...,...,...,...,...,...,...,...,...
6895505,2026-07-24,B6,BNA,JFK,11,765.0,5,False,False,1
6895506,2026-07-24,B6,BOS,SAN,6,2588.0,5,False,False,0
6895507,2026-07-24,B6,SRQ,JFK,11,1041.0,5,False,False,0
6895508,2026-07-24,B6,BOS,RSW,6,1249.0,5,False,False,0


Average Delay Rate: 0.2168239912638804


flight_date       0
carrier_code      0
origin_airport    0
dest_airport      0
dep_hour          0
distance          0
day_of_week       0
is_weekend        0
is_holiday        0
delayed           0
dtype: int64

Row count = 6895510. delay_rate = 0.2168. 0 NULLs in table.

In [51]:
import datetime

In [68]:
training_data = model_df[model_df["flight_date"] < datetime.datetime(2026, 6, 1)]
print(training_data.shape)
print("Average delayed rate of training data:", training_data["delayed"].mean())

testing_data = model_df[model_df["flight_date"] >= datetime.datetime(2026, 6, 1)]
print(testing_data.shape)
print("Average delayed rate of testing data:", testing_data["delayed"].mean())

assert(len(testing_data) + len(training_data) == len(model_df))


(5688277, 10)
Average delayed rate of training data: 0.20697515258135285
(1207233, 10)
Average delayed rate of testing data: 0.2632300475550287


Training data Rows  X Columns: (5688277, 10) \
Average delayed rate of training data: 0.20697515258135285 \
Testing data Rows  X Columns: (1207233, 10) \
Average delayed rate of testing data: 0.2632300475550287

# First Model

set up X and y

In [94]:
X_train = training_data.loc[:, ['carrier_code', 'origin_airport', 'dest_airport', 'dep_hour', 'distance', 'day_of_week', 'is_holiday']]
y_train = training_data["delayed"]
assert(X_train.size/len(x_train) + y_train.size/len(y_train) == 8)

Sort Columns into Groups

In [101]:
categorical_cols = X_train.loc[:, ['carrier_code', 'origin_airport', 'dest_airport', 'dep_hour', 'day_of_week']]
numeric_cols = X_train['is_holiday']
passthrough_cols = X_train['distance']

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [113]:
encoder = OneHotEncoder()

In [127]:
sample = X_train[["carrier_code"]].sample(5, random_state=42) #Randomly samples 5 to avoid the same airline appearing, then random_state=42 makes it the same random sample every time
print(sample)
encoder.fit(sample)
encoder.transform(sample).toarray()
encoder.categories_

        carrier_code
106926            MQ
1654696           UA
1186101           WN
3289332           DL
1244724           HA


[array(['DL', 'HA', 'MQ', 'UA', 'WN'], dtype=object)]

In [128]:
from sklearn.preprocessing import StandardScaler

In [129]:
scaler = StandardScaler()

# Scaling distance:
 Distances are in the thousands, but our other columns are just 0 or 1. The scaler shrinks distance onto a similar scale by turning each value into a z-score: how far it is from the average, measured in standard deviations. It first fits (learns the average and spread) and then transforms (converts each distance using those numbers).

In [163]:
sample2 = X_train[["distance"]].sample(5, random_state=42) # random 5 distances
scaler.fit(sample2)
scaled = scaler.transform(sample2)
print(scaled)
sample2_mean = sample2.mean()
sample2_sd = sample2.std()
print("mean:", scaler.mean_, "scale:", scaler.scale_, "sample2 SD:", sample2_sd)
chosen_distance_z_score = (sample2.iloc[0]-sample2_mean)/sample2_sd
print("Chosen Distance Z-Score:", chosen_distance_z_score, "Distance Z-score scaled from sample2:", scaled[0])

[[-0.28159214]
 [-0.48177613]
 [ 1.86371292]
 [ 0.00200184]
 [-1.10234649]]
mean: [431.4] scale: [299.72427329] sample2 SD: distance    335.101925
dtype: float64
Chosen Distance Z-Score: distance   -0.251864
dtype: float64 Distance Z-score scaled from sample2: [-0.28159214]


The pandas built in `.std()` uses the sample standard deviation formula: $$s = \sqrt{\frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n - 1}}$$ while the scaler uses the population standard deviation: $$\sigma = \sqrt{\frac{\sum_{i=1}^{N} (x_i - \mu)^2}{N}}$$ 
#### What does this mean?
    Although it makes a noticable difference in the Z-score here we will not have to worry about it in the future. It is noticable here because with only 5 values the sample sd formula divides by 4 while the population sd formula divides by 5. When we run our real training data we will have about 5,000,000 flights so dividing by 4,999,999 makes a very small difference in comparison to dividing by 5,000,000.